In [1]:
from utils.data import load


data = load(f'/home/burger/bioinfo/project/data/pdb.hdf5')

In [2]:
print('seq:', data[0].shape)
print('node_pos:', data[1].shape)
print('node_idx:', data[2].shape)
print('edge_nho:', data[3].shape)
print('edge_idx:', data[4].shape)
print('lab:', data[5].shape)

seq: (26041138,)
node_pos: (26041138, 5, 3)
node_idx: (106974,)
edge_nho: (2, 18611806)
edge_idx: (106974,)
lab: (106973,)


In [6]:
import numpy as np
from torch_geometric.data import Data
import torch as pt


seqs, node_pos, node_idx, edge_nho, edge_idx, lab = data
seq_list, graph_list, lab_list = [], [], []

for i in range(len(data[5])):
    seq = seqs[node_idx[i]: node_idx[i+1]]
    len_seq = len(seq)
    # shape:(len_seq,5,3) 0:N, 1:α, 2:C, 3:O, 4:β 
    pos = node_pos[node_idx[i]: node_idx[i+1]]
    # N-α-β这个夹角反映了氨基酸的空间结构，在化学键确定的前提下，N和β之间的距离就能反应角度 
    node_attr = np.sqrt(np.sum((pos[:,0] - pos[:,4])**2, axis=1))
    # 连接关系(肽键), 单向, 后续在embeddingBlock中搞双向
    tai = np.stack((np.arange(0, len_seq-1), np.arange(1, len_seq)), axis=0)
    # 连接关系(氢键), 单向
    nho = np.stack((edge_nho[0][edge_idx[i] : edge_idx[i+1]],
                    edge_nho[1][edge_idx[i] : edge_idx[i+1]]), axis=0)
    edge = np.concatenate((tai, nho), axis=1)
    # 边长 
    # 肽键：羧基碳接氨基氮
    tai_len = np.sqrt(np.sum((pos[edge[0][:], 2] - pos[edge[1][:], 0])**2, axis=1))
    # 氢键：氨基的氢和羧基的氧之间吸引产生，用氨基的氮坐标代替氢坐标 
    nho_len = np.sqrt(np.sum((pos[nho[0][:], 0] - pos[nho[1][:], 3])**2, axis=1))
    edge_len = np.concatenate((tai_len, nho_len), axis=0)
    graph = Data(x=node_attr, edge_index=edge, edge_attr=edge_len)
    seq_list.append(seq)
    graph_list.append(graph)
    lab_list.append(lab[i])
pt.save((seq_list, graph_list, lab_list), f'/home/burger/bioinfo/project/data/engineered_data.pt')